# Matchups con verificacion de geometria

Mismo algoritmo que `06_Datos_Satelitales/matchups.ipynb` (escena de menor diferencia temporal, +- 5 dias), con una correccion:
solo compiten las escenas cuya geometria **contiene el punto**. Antes se elegia la mas cercana en fecha aunque el punto cayera fuera
de la escena (en RN-UPM2 la orbita R024 deja los puntos fuera del swath y se elegia igual).

Salidas (en `data/nuevos/`):
- `registros_limpios_matchups_geometria.json`: los puntos con `pass_id` y `delta`
- `matchup_items_geometria.json`: las escenas elegidas

In [1]:
import time

import geopandas as gpd
import pandas as pd
from dotenv import load_dotenv
from pystac_client import Client
from pystac_client.exceptions import APIError
from pystac_client.stac_api_io import StacApiIO
from pystac import ItemCollection
from pyproj import Transformer
from shapely.geometry import shape
from shapely.prepared import prep
from tqdm import tqdm
from urllib3.util.retry import Retry

load_dotenv()

True

In [2]:
FILE = "../../data/registros_limpios_totales_nuevo.geojson"
MATCHUPS_ANTERIORES = "../../data/nuevos/registros_limpios_matchups_nuevos.json"
OUT_MATCHUPS = "../../data/nuevos/registros_limpios_matchups_geometria.json"
OUT_ITEMS = "../../data/nuevos/matchup_items_geometria.json"

CATALOG_URL = "https://stac.dataspace.copernicus.eu/v1"
COLLECTION = "sentinel-2-l1c"

CLUSTERS = [
    "RN-UPM1",
    "RN-UPM2",
    "RDP-MONTES",
    "LDS"
]

In [3]:
to_wgs84 = Transformer.from_crs("EPSG:32721", "EPSG:4326", always_xy=True)
gdf = gpd.read_file(FILE)
assert gdf.crs.to_epsg() == 32721

# la API tira Rate Limit a veces: reintenta en vez de crashear
retry = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[429, 502, 503, 504],
    respect_retry_after_header=True,
)
stac_io = StacApiIO(max_retries=retry)
catalogo = Client.open(CATALOG_URL, stac_io=stac_io)

#### Algoritmo de matchups
Igual que el original, con un paso mas:
1. para cada cluster, buscar las escenas de cada mes con muestras (+- 5 dias)
2. para cada punto, quedarse solo con las escenas cuya geometria contiene el punto
3. de esas, elegir la de menor diferencia temporal (maximo 5 dias)
4. si ninguna contiene el punto o ninguna esta a menos de 5 dias, el punto no tiene matchup

In [ ]:
def compute_matchups(cluster_points, items):
    puntos_wgs84 = cluster_points.to_crs("EPSG:4326").geometry
    # geometrias preparadas: contains() mucho mas rapido
    geometrias = {item.id: prep(shape(item.geometry)) for item in items}

    matchups = []
    sin_cobertura = []   # puntos que tenian escena en fecha pero ninguna los contiene
    for index, fila in cluster_points.iterrows():
        fecha = fila["fecha"]
        punto = puntos_wgs84.loc[index]
        best_diff = pd.Timedelta(days=5)   # threshold AND reset, per sample
        best_item = None
        hubo_en_fecha = False

        for item in items:
            fecha_satelite = pd.Timestamp(item.datetime).tz_localize(None)
            diff = abs(fecha_satelite - fecha)
            if diff > pd.Timedelta(days=5):
                continue
            hubo_en_fecha = True

            # el cambio importante!!
            if not geometrias[item.id].contains(punto):
                continue

            if diff <= best_diff:
                best_diff = diff
                best_item = item

        if best_item is not None:
            matchups.append((best_item, index, best_diff))
        elif hubo_en_fecha:
            sin_cobertura.append(index)
    return matchups, sin_cobertura

In [5]:
MAX_RETRIES = 5
BASE_DELAY = 5  # seconds

all_matchups = []
sin_cobertura = []

for cluster in CLUSTERS:
    cluster_points = gdf[gdf["grupo_nombre"] == cluster]
    cluster_bounds = cluster_points.total_bounds
    xmin, ymin, xmax, ymax = cluster_bounds

    xmin, ymin = to_wgs84.transform(xmin, ymin)
    xmax, ymax = to_wgs84.transform(xmax, ymax)

    tol = 0.001
    bbox = [xmin - tol, ymin - tol, xmax + tol, ymax + tol]

    months_with_samples = cluster_points["fecha"].dt.to_period("M").unique()
    items_by_id = {}

    for month in tqdm(months_with_samples, desc=f"Searching months of {cluster}"):
        start = (month.start_time - pd.Timedelta(days=5)).strftime("%Y-%m-%d")
        end = (month.end_time + pd.Timedelta(days=5)).strftime("%Y-%m-%d")

        for attempt in range(MAX_RETRIES):
            try:
                search = catalogo.search(
                    collections=[COLLECTION],
                    datetime=f"{start}/{end}",
                    bbox=bbox,
                )

                # Force the request inside the try
                items = list(search.items())

                for item in items:
                    items_by_id[item.id] = item

                break  # success

            except APIError as e:
                if attempt == MAX_RETRIES - 1:
                    raise

                delay = BASE_DELAY * (2 ** attempt)

                print(
                    f"\nAPI error for {start}/{end}. "
                    f"Retrying in {delay}s "
                    f"({attempt + 1}/{MAX_RETRIES})..."
                )

                time.sleep(delay)

        # Small delay even after successful requests
        time.sleep(1)

    cluster_matchups, cluster_sin_cobertura = compute_matchups(
        cluster_points,
        list(items_by_id.values())
    )

    all_matchups.extend(cluster_matchups)
    sin_cobertura.extend(cluster_sin_cobertura)
    print(f"{cluster}: {len(cluster_points)} puntos, {len(cluster_matchups)} con matchup, {len(cluster_sin_cobertura)} sin escena que los contenga")


Searching months of RN-UPM1:   0%|          | 0/28 [00:00<?, ?it/s]


Searching months of RN-UPM1:   4%|▎         | 1/28 [00:02<01:16,  2.82s/it]


Searching months of RN-UPM1:   7%|▋         | 2/28 [00:04<00:50,  1.94s/it]


Searching months of RN-UPM1:  11%|█         | 3/28 [00:07<01:00,  2.43s/it]


Searching months of RN-UPM1:  14%|█▍        | 4/28 [00:08<00:50,  2.09s/it]

...


Searching months of LDS:  97%|█████████▋| 68/70 [03:28<00:07,  3.52s/it]


Searching months of LDS:  99%|█████████▊| 69/70 [03:31<00:03,  3.21s/it]


Searching months of LDS: 100%|██████████| 70/70 [03:33<00:00,  2.96s/it]


Searching months of LDS: 100%|██████████| 70/70 [03:33<00:00,  3.06s/it]

LDS: 1154 puntos, 1140 con matchup, 0 sin escena que los contenga


In [6]:
gdf_copy = gdf.copy()
gdf_copy["pass_id"] = None
gdf_copy["delta"] = None

for item, index, delta in all_matchups:
    gdf_copy.loc[index, "delta"] = str(delta)
    gdf_copy.loc[index, "pass_id"] = item.id

gdf_copy = gdf_copy[gdf_copy["pass_id"].notna()].copy()
print(gdf_copy.shape, "| puntos sin escena que los contenga:", len(sin_cobertura))

(3055, 8) | puntos sin escena que los contenga: 6


#### Comparacion con los matchups anteriores

In [7]:
anteriores = gpd.read_file(MATCHUPS_ANTERIORES)
nuevos = gpd.read_file(OUT_MATCHUPS)

# los dos archivos tienen indices distintos: se cruzan por coordenadas, fecha y chla
def clave(d):
    return d.geometry.x.round(1).astype(str) + "_" + d.geometry.y.round(1).astype(str) + "_" + d["fecha"].astype(str) + "_" + d["chla"].astype(str)

anteriores["clave"] = clave(anteriores)
nuevos["clave"] = clave(nuevos)
anteriores = anteriores.drop_duplicates("clave").set_index("clave")
nuevos = nuevos.drop_duplicates("clave").set_index("clave")

comp = anteriores[["grupo_nombre", "pass_id"]].join(nuevos[["pass_id"]], rsuffix="_nuevo", how="left")
comp["estado"] = "igual"
comp.loc[comp["pass_id_nuevo"].isna(), "estado"] = "sin matchup ahora"
comp.loc[comp["pass_id_nuevo"].notna() & (comp["pass_id"] != comp["pass_id_nuevo"]), "estado"] = "cambio de escena"
print("antes:", len(anteriores), " ahora:", len(nuevos))
comp.groupby(["grupo_nombre", "estado"]).size().unstack(fill_value=0)

antes: 3059  ahora: 3053


estado,cambio de escena,igual,sin matchup ahora
grupo_nombre,,,
LDS,0,1140,0
RDP-MONTES,163,192,6
RN-UPM1,0,172,0
RN-UPM2,853,533,0


In [8]:
def matchup_items(matchups):
    items_by_ids = {}
    for m in matchups:
        item = m[0]
        items_by_ids[item.id] = item
    return items_by_ids

items_by_ids = matchup_items(all_matchups)
ic = ItemCollection(list(items_by_ids.values()))
ic.save_object(OUT_ITEMS)
gdf_copy.to_file(OUT_MATCHUPS, driver="GeoJSON")
print(len(items_by_ids), "escenas ->", OUT_ITEMS)

536 escenas -> ../../data/nuevos/matchup_items_geometria.json
